### Data Ingestion

In [2]:
### Document Stucture

from langchain_core.documents import Document

In [3]:
doc = Document(
    page_content="this is main text context I am using to create RAG",
    metadata={
        "source": "example.txt",
        "pages": 1,
        "author": "Dang Nguyen",
        "date_created": "2026-06-02"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Dang Nguyen', 'date_created': '2026-06-02'}, page_content='this is main text context I am using to create RAG')

In [4]:
### create a simple txt file

import os
os.makedirs("../data/text_files", exist_ok=True)

In [5]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}

for filepath,content in sample_texts.items():
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [6]:
### TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()
print(document)

C:\Users\84395\AppData\Local\Temp\ipykernel_30748\1962437515.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\Dev\rag-training\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]


In [7]:
### Directory Loader
from langchain_community.document_loaders import DirectoryLoader

## Load all the text files from the directory
dir_loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt", ## Patern to match files
    loader_cls=TextLoader, ## Loader class to use
    loader_kwargs={'encoding': 'utf-8'},
    show_progress=False
)

documents = dir_loader.load()
documents

[Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n\n\n    '),
 Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popu

In [8]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

dir_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf", ## Patern to match files
    loader_cls=PyMuPDFLoader, ## Loader class to use
    show_progress=False
)

pdf_documents = dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2022-07-05T08:32:50+07:00', 'source': '..\\data\\pdf\\BAI GIANG TRIET.pdf', 'file_path': '..\\data\\pdf\\BAI GIANG TRIET.pdf', 'total_pages': 167, 'format': 'PDF 1.5', 'title': 'BÀI GIẢNG MÔN TRIẾT HỌC MÁC - LÊNIN', 'author': 'DELL', 'subject': '', 'keywords': '', 'moddate': '2022-07-05T08:32:50+07:00', 'trapped': '', 'modDate': "D:20220705083250+07'00'", 'creationDate': "D:20220705083250+07'00'", 'page': 0}, page_content='3 \nHỌC VIỆN CÔNG NGHỆ BƯU CHÍNH VIỄN THÔNG \nKHOA CƠ BẢN I \nBỘ MÔN LÝ LUẬN CHÍNH TRỊ \n\uf0be\uf0be\uf0be\uf0be\uf0be\uf0be\uf0be\uf0be\uf0be \n \n \n \n \nBÀI GIẢNG \nTRIẾT HỌC MÁC - LÊNIN \n \n \n                               \nTs. Phạm Minh Ái \nThs. Phạm Thị Khánh \n(Đồng chủ biên) \n                                                              \n \n \n \n \nHÀ NỘI - 2021'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® W

In [9]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: BAI GIANG TRIET.pdf
  ✓ Loaded 167 pages

Processing: Chuong 4.pdf
  ✓ Loaded 53 pages

Total documents loaded: 220


In [10]:
all_pdf_documents


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2022-07-05T08:32:50+07:00', 'title': 'BÀI GIẢNG MÔN TRIẾT HỌC MÁC - LÊNIN', 'author': 'DELL', 'moddate': '2022-07-05T08:32:50+07:00', 'source': '..\\data\\pdf\\BAI GIANG TRIET.pdf', 'total_pages': 167, 'page': 0, 'page_label': '1', 'source_file': 'BAI GIANG TRIET.pdf', 'file_type': 'pdf'}, page_content='3 \nHỌC VIỆN CÔNG NGHỆ BƯU CHÍNH VIỄN THÔNG \nKHOA CƠ BẢN I \nBỘ MÔN LÝ LUẬN CHÍNH TRỊ \n\uf0be\uf0be\uf0be\uf0be\uf0be\uf0be\uf0be\uf0be\uf0be \n \n \n \n \nBÀI GIẢNG \nTRIẾT HỌC MÁC - LÊNIN \n \n \n                               \nTs. Phạm Minh Ái \nThs. Phạm Thị Khánh \n(Đồng chủ biên) \n                                                              \n \n \n \n \nHÀ NỘI - 2021'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2022-07-05T08:32:50+07:00', 'title': 'BÀI GIẢNG MÔN TRIẾT HỌC MÁC - LÊNIN', 'author': 'DELL', 

In [11]:
### Text splitting get into chunks

def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    text_spliter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_spliter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

chunks = split_documents(all_pdf_documents)
chunks

Split 220 documents into 636 chunks

Example chunk:
Content: 3 
HỌC VIỆN CÔNG NGHỆ BƯU CHÍNH VIỄN THÔNG 
KHOA CƠ BẢN I 
BỘ MÔN LÝ LUẬN CHÍNH TRỊ 
 
 
 
 
 
BÀI GIẢNG 
TRIẾT HỌC MÁC - LÊNIN 
 
 
                               
Ts. Phạm Minh Ái 
Ths. Phạ...
Metadata: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2022-07-05T08:32:50+07:00', 'title': 'BÀI GIẢNG MÔN TRIẾT HỌC MÁC - LÊNIN', 'author': 'DELL', 'moddate': '2022-07-05T08:32:50+07:00', 'source': '..\\data\\pdf\\BAI GIANG TRIET.pdf', 'total_pages': 167, 'page': 0, 'page_label': '1', 'source_file': 'BAI GIANG TRIET.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2022-07-05T08:32:50+07:00', 'title': 'BÀI GIẢNG MÔN TRIẾT HỌC MÁC - LÊNIN', 'author': 'DELL', 'moddate': '2022-07-05T08:32:50+07:00', 'source': '..\\data\\pdf\\BAI GIANG TRIET.pdf', 'total_pages': 167, 'page': 0, 'page_label': '1', 'source_file': 'BAI GIANG TRIET.pdf', 'file_type': 'pdf'}, page_content='3 \nHỌC VIỆN CÔNG NGHỆ BƯU CHÍNH VIỄN THÔNG \nKHOA CƠ BẢN I \nBỘ MÔN LÝ LUẬN CHÍNH TRỊ \n\uf0be\uf0be\uf0be\uf0be\uf0be\uf0be\uf0be\uf0be\uf0be \n \n \n \n \nBÀI GIẢNG \nTRIẾT HỌC MÁC - LÊNIN \n \n \n                               \nTs. Phạm Minh Ái \nThs. Phạm Thị Khánh \n(Đồng chủ biên) \n                                                              \n \n \n \n \nHÀ NỘI - 2021'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2022-07-05T08:32:50+07:00', 'title': 'BÀI GIẢNG MÔN TRIẾT HỌC MÁC - LÊNIN', 'author': 'DELL', 

### Embedding and VectorStoreDB

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [13]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-V2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dismention: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts ...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    
## initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-V2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3556.69it/s]


Model loaded successfully. Embedding dismention: 384


C:\Users\84395\AppData\Local\Temp\ipykernel_30748\1884678464.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dismention: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [14]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or Create collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing document in collection: {self.collection.count()}")

        except Exception as e:
            raise

    def add_documents(self, documents: List[any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store ...")
        
        # Prepare data for chromaDB
        ids = []    
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore 

Vector store initialized. Collection: pdf_documents
Existing document in collection: 636


In [15]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

### Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

### Store int he vector database
vectorstore.add_documents(chunks, embeddings) 

Generating embeddings for 636 texts ...


Batches: 100%|██████████| 20/20 [00:22<00:00,  1.13s/it]


Generated embeddings with shape: (636, 384)
Adding 636 documents to vector store ...
successfully added 636 documents to vector store
Total documents in collection: 1272


### Retriver Pipeline From VectorStore

In [16]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrive(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, any]]:
        """
         Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []
        
rag_retriever = RAGRetriever(vectorstore,embedding_manager)
rag_retriever

In [17]:
rag_retriever.retrive("Triết học là gì")

Retrieving documents for query: 'Triết học là gì'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts ...


Batches: 100%|██████████| 1/1 [00:00<00:00, 70.62it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_77c0e2ca_89',
  'content': 'BÀI GIẢNG MÔN TRIẾT HỌC MÁC - LÊNIN  \n \n  \n \nBỘ MÔN LÝ LUẬN CHÍNH TRỊ - PTIT Page 27 \nCÂU HỎI ÔN TẬP \n1. Hãy nêu và phân tích khái niệm và nguồn gốc ra đời của triết học? \n2. Vấn đề cơ b ản c ủa tri ết học là gì? Hãy phân bi ệt ch ủ nghĩa duy v ật và ch ủ \nnghĩa duy tâm, thuy ết Có th ể biết và thuy ết Không th ể biết? Hãy phân bi ệt phương \npháp biện chứng và siêu hình? \n3. Triết học Mác – Lênin ra đời dựa trên những tiền đề nào? \n4. Thực chất và ý nghĩa cuộc cách mạng trong triết học do C.Mác và Ph.Ăngghen \nthực hiện là gì? Hãy nêu nh ững nội dung chủ yếu mà V.I.Lênin b ổ sung và phát tri ển \ntriết học Mác? \n5. Hãy nêu đối tượng nghiên cứu và chức năng của triết học Mác – Lênin? \nVẤN ĐỀ THẢO LUẬN \n1. Cuộc đấu tranh giữa chủ nghĩa duy vật và chủ nghĩa duy tâm còn t ồn tại trong \ntriết học hiện đại không? Tại sao trong thời đại ngày nay, các tín ngư ỡng tôn giáo vẫn \ncó duy trì được sức sống và sự phát triển của mình?',
  'meta